# Customer Dataset Dimensionality Reduction using PCA
**Aim:** Apply PCA to customer spending features, compare PCA without scaling and PCA after StandardScaler, and interpret the transformed data.

Dataset: UCI Wholesale Customers (440 records). PCA uses the six annual spending features; `Channel` and `Region` are retained only as descriptive columns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error


## 1. Load the dataset
The dataset is downloaded from the UCI Machine Learning Repository. Internet access is required for this cell.

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv'
df = pd.read_csv(url)
print('Shape:', df.shape)
display(df.head())
display(df.info())


## 2. Select numerical customer spending features
PCA is applied to the six spending columns. `Channel` and `Region` are categorical descriptors and are not included in this PCA experiment.

In [ ]:
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X = df[features].copy()
print('Missing values:', X.isnull().sum().sum())
display(X.describe().round(2))


## 3. Exploratory data analysis

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=X)
plt.xticks(rotation=30, ha='right')
plt.title('Distribution of customer spending features')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,6))
sns.heatmap(X.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation between spending features')
plt.tight_layout()
plt.show()


## 4. PCA without feature scaling
PCA is variance-based. Features with larger numerical scales can dominate when the original data is used directly.

In [ ]:
pca_raw = PCA()
X_raw_pca = pca_raw.fit_transform(X)
raw_variance = pca_raw.explained_variance_ratio_
raw_table = pd.DataFrame({'Principal Component':[f'PC{i+1}' for i in range(len(features))], 'Explained Variance Ratio':raw_variance, 'Cumulative Variance':np.cumsum(raw_variance)})
display(raw_table.round(4))


## 5. StandardScaler followed by PCA
StandardScaler transforms each feature to approximately zero mean and unit standard deviation before PCA.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca_scaled = PCA()
X_scaled_pca = pca_scaled.fit_transform(X_scaled)
scaled_variance = pca_scaled.explained_variance_ratio_
scaled_table = pd.DataFrame({'Principal Component':[f'PC{i+1}' for i in range(len(features))], 'Explained Variance Ratio':scaled_variance, 'Cumulative Variance':np.cumsum(scaled_variance)})
display(scaled_table.round(4))


## 6. Compare explained variance

In [ ]:
comparison = pd.DataFrame({'Component':[f'PC{i+1}' for i in range(len(features))], 'PCA without scaling':raw_variance, 'PCA after StandardScaler':scaled_variance})
display(comparison.round(4))
comparison.set_index('Component').plot(kind='bar', figsize=(9,5))
plt.ylabel('Explained variance ratio')
plt.title('PCA comparison: without scaling vs with StandardScaler')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(range(1,7), np.cumsum(raw_variance), marker='o', label='Without scaling')
plt.plot(range(1,7), np.cumsum(scaled_variance), marker='o', label='With StandardScaler')
plt.axhline(0.90, linestyle='--', label='90% variance')
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance')
plt.title('Cumulative explained variance')
plt.xticks(range(1,7))
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 7. Visualize customers in the first two principal components

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
axes[0].scatter(X_raw_pca[:,0], X_raw_pca[:,1], alpha=0.6)
axes[0].set_title('PCA without scaling')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[1].scatter(X_scaled_pca[:,0], X_scaled_pca[:,1], alpha=0.6)
axes[1].set_title('PCA after StandardScaler')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.tight_layout(); plt.show()


## 8. Reduce the data to two dimensions
The following creates a two-dimensional representation using the scaled PCA result. The amount of variance retained is printed for interpretation.

In [ ]:
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)
reduced_df = pd.DataFrame(X_2d, columns=['PC1','PC2'])
reduced_df['Channel'] = df['Channel'].values
print('Variance retained by 2 components:', round(pca_2d.explained_variance_ratio_.sum()*100, 2), '%')
display(reduced_df.head())


## 9. Conclusion
PCA converts the six original spending features into principal components that summarize variation in the customer data. The comparison demonstrates why scaling matters: PCA without scaling can be dominated by high-variance spending features, while StandardScaler gives each feature comparable initial scale. The exact variance percentages and plots should be taken from the executed notebook output.